# The 230 space groups

**Professor Nano Labs · Symmetry**

**Try it live in your browser:** [The space groups lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/space-groups/)
**Watch first:** [Why Can Quasicrystals Have "Forbidden" Fivefold Symmetry?](https://www.youtube.com/watch?v=KI25B8Vz4rk)
**Watch later:** [Basics of crystal structure](https://www.youtube.com/watch?v=rpZOt_rBx4I) · [Deriving the Bloch's theorem](https://www.youtube.com/watch?v=Bj2rNT5fa9s) · [Spatial vs momentum translation operators](https://www.youtube.com/watch?v=cu-XQdRFKbE)

A symmetry operation acts on fractional coordinates as $x\to Wx+w$. This notebook

1. loads the operations of any space group from spglib, in the standard settings of the International Tables;
2. classifies each one from the determinant and trace of $W$, and finds its axis or plane, its screw or glide part and its location;
3. counts the general positions, the symmorphic groups and the groups with no mirror-type operation;
4. checks the crystallographic restriction;
5. builds crystals from their Wyckoff positions and checks atom counts and bond lengths.

It checks every number quoted in the lab's "Predict first" answers. It needs NumPy, Matplotlib and spglib (`pip install spglib`). **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spglib

# one Hall number per space group: the standard setting (origin choice 2 where there are two, hexagonal axes for R groups)
HALL = {}
for h in range(1, 531):
    t = spglib.get_spacegroup_type(h)
    n, ch = t.number, t.choice
    if n not in HALL or ch in ('2', 'H'):
        HALL[n] = h

def operations(n):
    # every operation of space group n in one conventional cell, including the centring translations
    s = spglib.get_symmetry_from_database(HALL[n])
    return [(np.array(W, float), np.array(w, float) % 1) for W, w in zip(s['rotations'], s['translations'])]

def symbol(n): return spglib.get_spacegroup_type(HALL[n]).international_short
print(len(HALL), 'space groups;', 'No. 14 is', symbol(14), 'with', len(operations(14)), 'operations;', 'No. 225 is', symbol(225), 'with', len(operations(225)))

## 1 · Classifying an operation

The type of $W$ follows from $\det W$ and $\mathrm{tr}\,W$. The intrinsic (screw or glide) part is $w$ averaged over one cycle of $W$, and the element sits at the fixed points of what remains: $(W-1)x_0=-(w-w_{\rm intr})$.

In [ ]:
def classify(W, w):
    d, t = round(np.linalg.det(W)), round(np.trace(W))
    kind = ({3: '1', -1: '2', 0: '3', 1: '4', 2: '6'} if d > 0 else {-3: '-1', 1: 'm', 0: '-3', -1: '-4', -2: '-6'})[t]
    k = {'1': 1, '2': 2, '3': 3, '4': 4, '6': 6, '-1': 2, 'm': 2, '-3': 6, '-4': 4, '-6': 6}[kind]
    tin = np.zeros(3)
    if d > 0 or kind == 'm':
        v = np.zeros(3)
        for _ in range(k): v = W @ v + w            # the translation part of (W, w)^k
        tin = v / k
    x0 = np.linalg.lstsq(W - np.eye(3), -(w - tin), rcond=None)[0]
    return dict(kind=kind, tin=tin, x0=x0, det=d)

def axis(W):
    # the direction left alone by the rotation part
    R = W if round(np.linalg.det(W)) > 0 else -W
    vals, vecs = np.linalg.eig(R)
    u = np.real(vecs[:, np.argmin(abs(vals - 1))]); u = u / u[np.argmax(abs(u))]
    return np.round(u, 6)

for W, w in operations(14):
    c = classify(W, w)
    print(f"{c['kind']:>3}  shift {np.round(w, 3)}  intrinsic {np.round(c['tin'], 3)}  through {np.round(c['x0'], 3)}")

### Questions 1 to 3

1. P2₁/c (No. 14): four operations: the identity, a 2₁ screw along $b$ at $0,y,\tfrac14$, an inversion centre at the origin, and a $c$-glide at $y=\tfrac14$.
2. P2 (No. 3) against P2₁ (No. 4): the second copy of $(x,y,z)$ is at $(-x,y,-z)$ and $(-x,y+\tfrac12,-z)$.
3. Pnma (No. 62): an $n$-glide perpendicular to $a$ at $x=\tfrac14$, a mirror at $y=\tfrac14$, an $a$-glide at $z=\tfrac14$; eight operations.

In [ ]:
ops14 = [classify(W, w) for W, w in operations(14)]
kinds = sorted(c['kind'] for c in ops14)
print('P2_1/c:', kinds)
assert kinds == ['-1', '1', '2', 'm'] and len(ops14) == 4
screw = next(c for c in ops14 if c['kind'] == '2'); glide = next(c for c in ops14 if c['kind'] == 'm')
assert np.allclose(screw['tin'], [0, 0.5, 0]) and np.isclose(screw['x0'][2] % 1, 0.25)
assert np.allclose(glide['tin'], [0, 0, 0.5]) and np.isclose(glide['x0'][1] % 1, 0.25)

x = np.array([0.137, 0.229, 0.083])
for n in (3, 4):
    second = [(W @ x + w) % 1 for W, w in operations(n) if round(np.trace(W)) != 3][0]
    print(f'{symbol(n)}: second copy of {x} is at {np.round(second, 3)}')
assert np.allclose([(W @ x + w) % 1 for W, w in operations(4) if round(np.trace(W)) != 3][0], [1 - 0.137, 0.729, 1 - 0.083])

ops62 = [classify(W, w) for W, w in operations(62)]
planes = {}
for (W, w), c in zip(operations(62), ops62):
    if c['kind'] == 'm':
        nrm = int(np.argmax(abs(axis(W))))
        planes['abc'[nrm]] = (np.round(c['tin'], 3), round(c['x0'][nrm] % 1, 3))
print('Pnma planes, by the axis they are perpendicular to (glide vector, position):', planes)
assert len(ops62) == 8 and np.allclose(planes['a'][0], [0, 0.5, 0.5]) and planes['a'][1] == 0.25
assert np.allclose(planes['b'][0], [0, 0, 0]) and planes['b'][1] == 0.25 and np.allclose(planes['c'][0], [0.5, 0, 0]) and planes['c'][1] == 0.25

## 2 · All 230 groups (Questions 4 and 7)

The number of general positions in one cell; the symmorphic groups (an origin that every rotation part leaves in place); and the groups with no mirror-type operation ($\det W=+1$ throughout), the only ones a chiral crystal can have.

In [ ]:
counts = {n: len(operations(n)) for n in range(1, 231)}
print('Fm-3m (225):', counts[225], '| Fd-3m (227):', counts[227], '| F-43m (216):', counts[216], '| largest:', max(counts.values()))
assert counts[225] == 192 and counts[227] == 192 and counts[216] == 96

def symmorphic(n):
    # symmorphic: there is an origin that every rotation part leaves fixed, so each one appears with no shift at all.
    # The standard settings put the origin there whenever such a point exists.
    by_W = {}
    for W, w in operations(n): by_W.setdefault(W.astype(int).tobytes(), []).append(w)
    return all(any(np.allclose((w + 0.5) % 1 - 0.5, 0, atol=1e-6) for w in ws) for ws in by_W.values())
sym = [n for n in range(1, 231) if symmorphic(n)]
sohncke = [n for n in range(1, 231) if all(round(np.linalg.det(W)) > 0 for W, w in operations(n))]
print(f'{len(sym)} symmorphic groups; {len(sohncke)} groups with rotations only (Sohncke groups)')
assert len(sym) == 73 and len(sohncke) == 65

q = [classify(W, w)['kind'] for W, w in operations(152)]
print('alpha-quartz, P3_121:', sorted(q), '- improper operations:', sum(1 for W, w in operations(152) if round(np.linalg.det(W)) < 0))
assert len(q) == 6 and all(round(np.linalg.det(W)) > 0 for W, w in operations(152))

## 3 · The crystallographic restriction (Question 8)

In cell coordinates $W$ has whole-number entries, so its trace is a whole number; a rotation by $2\pi/n$ has trace $1+2\cos(2\pi/n)$.

In [ ]:
for n in range(1, 13):
    tr = 1 + 2 * np.cos(2 * np.pi / n)
    print(f'n = {n:2d}: trace {tr:7.4f}', 'allowed' if abs(tr - round(tr)) < 1e-9 else '')
allowed = [n for n in range(1, 13) if abs(1 + 2 * np.cos(2 * np.pi / n) - round(1 + 2 * np.cos(2 * np.pi / n))) < 1e-9]
assert allowed == [1, 2, 3, 4, 6] and round(1 + 2 * np.cos(2 * np.pi / 5), 3) == 1.618
orders = {abs(int(classify(W, w)['kind'])) for n in range(1, 231) for W, w in operations(n) if classify(W, w)['kind'] not in ('m',)}
print('rotation orders found in all 230 groups:', sorted(orders))
assert orders <= {1, 2, 3, 4, 6}

## 4 · Building crystals (Questions 5 and 6)

Each atom of the asymmetric unit is copied by every operation, brought into the cell, and counted once.

In [ ]:
def cell_matrix(a, b, c, al, be, ga):
    al, be, ga = np.radians([al, be, ga])
    cx, cy = c * np.cos(be), c * (np.cos(al) - np.cos(be) * np.cos(ga)) / np.sin(ga)
    return np.array([[a, b * np.cos(ga), cx], [0, b * np.sin(ga), cy], [0, 0, np.sqrt(c * c - cx * cx - cy * cy)]])

def build(n, atoms):
    out = []
    for el, f in atoms:
        for W, w in operations(n):
            y = (W @ np.array(f) + w) % 1
            if not any(e == el and np.allclose((y - z + 0.5) % 1 - 0.5, 0, atol=1e-5) for e, z in out): out.append((el, y))
    return out

def nearest(atoms, M, i):
    d = [np.linalg.norm(M @ ((z - atoms[i][1] + 0.5) % 1 - 0.5)) for j, (e, z) in enumerate(atoms) if j != i]
    return min(d)

CRYSTALS = {
    'diamond (Si)': (227, (5.431, 5.431, 5.431, 90, 90, 90), [('Si', (1/8, 1/8, 1/8))]),
    'zincblende (GaAs)': (216, (5.653, 5.653, 5.653, 90, 90, 90), [('Ga', (0, 0, 0)), ('As', (1/4, 1/4, 1/4))]),
    'rock salt (NaCl)': (225, (5.640, 5.640, 5.640, 90, 90, 90), [('Na', (0, 0, 0)), ('Cl', (1/2, 1/2, 1/2))]),
    'graphite': (194, (2.464, 2.464, 6.711, 90, 90, 120), [('C', (0, 0, 1/4)), ('C', (1/3, 2/3, 1/4))]),
    'alpha-quartz': (152, (4.913, 4.913, 5.405, 90, 90, 120), [('Si', (0.4697, 0, 1/3)), ('O', (0.4135, 0.2669, 0.2191))]),
}
built = {}
for name, (n, cell, atoms) in CRYSTALS.items():
    A = build(n, atoms); M = cell_matrix(*cell); built[name] = (A, M)
    els = {}
    for e, _ in A: els[e] = els.get(e, 0) + 1
    print(f'{name:18s} {symbol(n):8s} atoms in the cell {els}, nearest neighbour {nearest(A, M, 0):.3f} A')
assert len(built['diamond (Si)'][0]) == 8 and round(nearest(built['diamond (Si)'][0], built['diamond (Si)'][1], 0), 2) == 2.35
assert len(built['rock salt (NaCl)'][0]) == 8 and round(nearest(built['rock salt (NaCl)'][0], built['rock salt (NaCl)'][1], 0), 2) == 2.82
assert len(built['graphite'][0]) == 4 and round(nearest(built['graphite'][0], built['graphite'][1], 0), 2) == 1.42
assert len(built['alpha-quartz'][0]) == 9

# Question 5: zincblende keeps half of diamond's operations, and loses the inversion centres
print('operations in the cell: diamond', counts[227], 'zincblende', counts[216],
      '| inversions in zincblende:', sum(1 for W, w in operations(216) if classify(W, w)['kind'] == '-1'))
assert counts[216] * 2 == counts[227] and not any(classify(W, w)['kind'] == '-1' for W, w in operations(216))

# Question 6: the 6_3 screw maps every graphite atom onto an atom
A, M = built['graphite']
six3 = [(W, w) for W, w in operations(194) if classify(W, w)['kind'] == '6' and np.allclose(classify(W, w)['tin'][2] % 1, 0.5)]
W, w = six3[0]
ok = all(any(np.allclose(((W @ z + w) - y + 0.5) % 1 - 0.5, 0, atol=1e-5) for _, y in A) for _, z in A)
moved = [round(((W @ z + w) % 1)[2], 3) for _, z in A]
print('6_3 screw on graphite: every atom lands on an atom:', ok, '| heights before', [round(z[2], 3) for _, z in A], 'after', moved)
assert ok

In [ ]:
# a picture: diamond's cell, looking down c
A, M = built['diamond (Si)']
P = np.array([M @ z for _, z in A])
plt.figure(figsize=(4.2, 4.2))
plt.scatter(P[:, 0], P[:, 1], c=P[:, 2], cmap='viridis', s=120, edgecolor='k')
for p in P: plt.annotate(f'{p[2] / 5.431:.2f}', (p[0] + 0.15, p[1] + 0.15), fontsize=8)
plt.gca().set_aspect('equal'); plt.title('diamond: 8 atoms from 1, heights in units of c'); plt.xlabel('x (A)'); plt.ylabel('y (A)')
plt.tight_layout(); plt.show()
print('notebook OK: every assertion passed')